# IARA-Imagem — BATERIA (Colab, ~4-6h de GPU)

A rodada anterior revelou 2 coisas: **(a)** a pirâmide colapsava (loss só no nível final — corrigido
com supervisão progressiva, fiel ao M44); **(b)** o M44 batia JPEG treinando num **domínio estreito**
(frames de 1 vídeo aéreo), não em dados diversos.

Esta bateria responde 4 perguntas com número, em vez de só treinar mais:

| # | pergunta | como responde |
|---|---|---|
| 1 | a pirâmide vive com o loss novo? | `KB por grau` > 0 em todos |
| 2 | estava **sub-treinado**? | curva de escala 10k / 30k / 60k passos |
| 3 | qual a curva R-D **honesta** vs JPEG? | **varredura de λ** (4 modelos) = a curva de verdade |
| 4 | a vitória do M44 era do **domínio estreito**? | mesmo treino em diverso vs estreito |

Tudo com checkpoint (sobrevive a desconexão) e resumo final em JSON.

**Antes:** `Ambiente de execução → Alterar tipo → GPU` → `Executar tudo`.


## 1) Setup + dados (5000 imagens)


In [ ]:
import torch, time, os, glob, subprocess, json, io, math
import numpy as np, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
assert torch.cuda.is_available(), "Ligue a GPU!"
GPU=torch.cuda.get_device_name(0); print("GPU:",GPU)
T0=time.time()
def log(m): print(f"[{(time.time()-T0)/60:6.1f}min] {m}",flush=True)
DATA="/content/val2017"
if not os.path.isdir(DATA):
    print("baixando COCO val2017 …",flush=True)
    subprocess.run(["wget","-q","http://images.cocodataset.org/zips/val2017.zip","-O","/content/coco.zip"],check=True)
    subprocess.run(["unzip","-q","/content/coco.zip","-d","/content/"],check=True)
    os.remove("/content/coco.zip")
FILES=sorted(glob.glob(f"{DATA}/*.jpg")); print(len(FILES),"imagens")
os.makedirs("/content/out",exist_ok=True)


## 2) Arquitetura + loss PROGRESSIVO (a correção)


In [ ]:
LN2=float(np.log(2)); S=128; dev="cuda"
def chans(d,c): return [3]+[32,48,64,96,128][:d-1]+[c]
def enc_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.Conv2d(cs[i],cs[i+1],4,2,1),nn.GELU()]
    return nn.Sequential(*L[:-1])
def dec_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.ConvTranspose2d(cs[i],cs[i+1],4,2,1),nn.GELU()]
    return nn.Sequential(*L[:-1])
class PyramidN(nn.Module):
    def __init__(self,specs):
        super().__init__(); self.specs=specs
        self.encs=nn.ModuleList([enc_stack(chans(d,c)) for d,c in specs])
        self.decs=nn.ModuleList([dec_stack(list(reversed(chans(d,c)))) for d,c in specs])
        self.logb=nn.ParameterList([nn.Parameter(torch.zeros(c)) for _,c in specs])
    def _q(self,z,train): return z+(torch.rand_like(z)-0.5) if train else torch.round(z)
    def forward(self,x,train=True):
        rec=None; zs=[]; recs=[]
        for i in range(len(self.specs)):
            tgt=x if rec is None else (x-rec)
            z=self._q(self.encs[i](tgt),train); zs.append(z)
            out=self.decs[i](z)
            rec=torch.sigmoid(out) if i==0 else (rec+out).clamp(0,1)
            recs.append(rec)
        return {"rec":rec,"recs":recs,"z":zs}
def rate(z,logb):
    b=F.softplus(logb)[None,:,None,None]+1e-3
    return (torch.log(2*b)+z.abs()/b).mean()/LN2
def kb_of(z):
    v=z.detach().cpu().numpy().ravel().astype(np.int32)
    if v.size==0: return 0.0
    p=np.bincount(v-v.min()).astype(np.float64); p=p[p>0]/v.size
    return v.size*float(-(p*np.log2(p)).sum())/8/1024
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m<1e-12 else 10*np.log10(1.0/m)
SPECS=[(5,24),(4,16),(3,12),(2,8)]; MULT=2**max(d for d,_ in SPECS)
print("PyramidN ·",SPECS,"· múltiplo",MULT)


## 3) Carregar imagens NA GPU (treino muito mais rápido)


In [ ]:
N_IMGS=2500; HELD=80
def load(fs,cap=288):
    out=[]
    for f in fs:
        try:
            im=Image.open(f).convert("RGB")
            if min(im.size)<S: continue
            if min(im.size)>cap:
                r=cap/min(im.size); im=im.resize((int(im.width*r),int(im.height*r)))
            w,h=im.size; im=im.crop((0,0,w//MULT*MULT,h//MULT*MULT))
            if min(im.size)<S: continue
            out.append(torch.from_numpy(np.asarray(im)).float().permute(2,0,1)/255.)
        except Exception: pass
    return out
ALL=load(FILES[:N_IMGS]); pool_c=ALL[:-HELD]; held=ALL[-HELD:]
pool=[t.to(dev) for t in pool_c]     # <- pool na GPU: elimina o gargalo de transferência
log(f"treino {len(pool)} · held-out {len(held)} (o original tinha 451 no total)")
def crops(idxs,n):
    out=[]
    for _ in range(n):
        im=pool[idxs[int(torch.randint(0,len(idxs),(1,)))]]
        i=int(torch.randint(0,im.shape[1]-S+1,(1,))); j=int(torch.randint(0,im.shape[2]-S+1,(1,)))
        out.append(im[:,i:i+S,j:j+S])
    return torch.stack(out)


## 4) Treinador (loss progressivo + cosine LR + checkpoint)


In [ ]:
def treinar(steps, lam, idxs=None, B=32, lr=6e-4, tag="run", log_every=2000):
    idxs=idxs or list(range(len(pool)))
    m=PyramidN(SPECS).to(dev); opt=torch.optim.AdamW(m.parameters(),lr=lr)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps,pct_start=0.05)
    hist=[]
    for s in range(steps):
        x=crops(idxs,B); o=m(x,train=True)
        dist=sum(F.mse_loss(r,x) for r in o["recs"])          # <- PROGRESSIVO (M44): cada grau supervisionado
        loss=dist+lam*sum(rate(z,lb) for z,lb in zip(o["z"],m.logb))
        opt.zero_grad(); loss.backward(); opt.step(); sch.step(); hist.append(loss.item())
        if s%log_every==0 or s==steps-1:
            with torch.no_grad(): kbs=[round(kb_of(z),2) for z in m(x,train=False)["z"]]
            log(f"  [{tag}] {s}/{steps} loss {np.mean(hist[-200:]):.4f} | KB/grau {kbs}")
    torch.save(m.state_dict(),f"/content/out/{tag}.pt"); m.eval()
    return m,hist
# --- avaliação: pontos R-D do modelo + JPEG na MESMA banda ---
_JC={}
def jpeg_pts(img_t,key):
    if key in _JC: return _JC[key]
    arr=(img_t.permute(1,2,0).cpu().numpy()*255).astype(np.uint8); ref=arr.astype(np.float32)/255; pts=[]
    for q in range(2,96,2):
        b=io.BytesIO(); Image.fromarray(arr).save(b,"JPEG",quality=q)
        rec=np.asarray(Image.open(io.BytesIO(b.getvalue())).convert("RGB"),np.float32)/255
        pts.append((len(b.getvalue())/1024,psnr(rec,ref)))
    _JC[key]=pts; return pts
def avaliar(m,imgs,n=24):
    out=[]
    with torch.no_grad():
        for L in range(1,len(SPECS)+1):
            ps,kbs,pj=[],[],[]
            for i,im in enumerate(imgs[:n]):
                x=im[None].to(dev); o=m(x,train=False)
                rec=o["recs"][L-1][0].cpu(); kb=sum(kb_of(z) for z in o["z"][:L])
                ps.append(psnr(rec.numpy(),x[0].cpu().numpy())); kbs.append(kb)
                pts=jpeg_pts(im,i); pj.append(min(pts,key=lambda t:abs(t[0]-kb))[1])
            out.append({"nivel":L,"kb":float(np.mean(kbs)),"psnr":float(np.mean(ps)),"jpeg":float(np.mean(pj))})
    return out
RES={}


## 5) EXP-1 — está sub-treinado? (curva de escala: 10k / 30k / 60k)

Se o PSNR ainda sobe de 30k→60k, era fome de treino. Se estabiliza, o limite é a arquitetura.


In [ ]:
for steps in [10000,30000,60000]:
    m,h=treinar(steps,lam=1.2e-3,tag=f"scale{steps}")
    r=avaliar(m,held); RES[f"steps{steps}"]=r
    fin=r[-1]; log(f"EXP1 steps={steps}: nível4 {fin['psnr']:.2f} dB @ {fin['kb']:.2f} KB · JPEG {fin['jpeg']:.2f} · Δ {fin['psnr']-fin['jpeg']:+.2f}")
    json.dump(RES,open("/content/out/resultados.json","w"),indent=1)
xs=[10000,30000,60000]; ys=[RES[f"steps{s}"][-1]["psnr"] for s in xs]
plt.figure(figsize=(5,3)); plt.plot(xs,ys,"o-"); plt.xlabel("passos"); plt.ylabel("PSNR nível 4 (dB)")
plt.title("ainda sobe = era sub-treino"); plt.grid(alpha=.3); plt.show()


## 6) EXP-2 — 🏆 a curva R-D honesta (varredura de λ)

Um codec de verdade se compara treinando em vários pontos de taxa. Cada λ = um ponto da curva.


In [ ]:
BEST=60000
for lam in [4e-4,1.2e-3,4e-3,1.2e-2]:
    m,_=treinar(BEST,lam=lam,tag=f"lam{lam:.0e}")
    r=avaliar(m,held); RES[f"lam{lam:.0e}"]=r
    fin=r[-1]; log(f"EXP2 λ={lam:.0e}: {fin['psnr']:.2f} dB @ {fin['kb']:.2f} KB · JPEG {fin['jpeg']:.2f} · Δ {fin['psnr']-fin['jpeg']:+.2f}")
    json.dump(RES,open("/content/out/resultados.json","w"),indent=1)
pts=[(RES[k][-1]["kb"],RES[k][-1]["psnr"],RES[k][-1]["jpeg"]) for k in RES if k.startswith("lam")]
pts.sort()
plt.figure(figsize=(6,4))
plt.plot([p[0] for p in pts],[p[1] for p in pts],"o-",label="IARA pirâmide")
plt.plot([p[0] for p in pts],[p[2] for p in pts],"s--",label="JPEG")
plt.xlabel("KB"); plt.ylabel("PSNR (dB)"); plt.legend(); plt.grid(alpha=.3)
plt.title("curva R-D honesta (acima = melhor)"); plt.show()
print("Δ médio vs JPEG:",round(float(np.mean([p[1]-p[2] for p in pts])),2),"dB")


## 7) EXP-3 — a vitória do M44 era do DOMÍNIO ESTREITO?

Mesmo treino, dados estreitos (1 cluster) vs diversos. Se o estreito vira Δ positivo, hipótese confirmada.


In [ ]:
def feat(img):
    x=F.interpolate(img[None].cpu(),size=(S,S),mode="bilinear")[0]
    hist=torch.cat([torch.histc(x[c],bins=4,min=0,max=1) for c in range(3)])/(S*S)
    g=x.mean(0); fft=torch.fft.rfft2(g).abs(); h,w=fft.shape
    yy,xx=torch.meshgrid(torch.arange(h),torch.arange(w),indexing="ij")
    r=torch.sqrt((yy/h)**2+(xx/w)**2)
    bands=torch.tensor([fft[(r>=a)&(r<b)].mean() for a,b in [(0,.05),(.05,.15),(.15,.3),(.3,.5),(.5,1.)]])
    return torch.cat([hist,bands/(bands.sum()+1e-9)]).numpy()
Ftr=np.stack([feat(i) for i in pool]); mu,sd=Ftr.mean(0),Ftr.std(0)+1e-9; Fn=(Ftr-mu)/sd
K=4; rng=np.random.default_rng(0); cent=Fn[rng.choice(len(Fn),K,replace=False)].copy()
for _ in range(50):
    lab=((Fn[:,None]-cent[None])**2).sum(-1).argmin(1)
    for k in range(K):
        if (lab==k).any(): cent[k]=Fn[lab==k].mean(0)
big=int(np.bincount(lab).argmax()); idxN=[i for i in range(len(pool)) if lab[i]==big]
hl=((np.stack([feat(i) for i in held])-mu)/sd)
hlab=((hl[:,None]-cent[None])**2).sum(-1).argmin(1)
held_n=[held[i] for i in range(len(held)) if hlab[i]==big]
log(f"domínio estreito: {len(idxN)} treino / {len(held_n)} held-out")
m_n,_=treinar(BEST,lam=1.2e-3,idxs=idxN,tag="estreito")
r_n=avaliar(m_n,held_n,n=min(24,len(held_n))); RES["estreito"]=r_n
# o modelo DIVERSO avaliado nas MESMAS imagens (comparação justa)
m_d=PyramidN(SPECS).to(dev); m_d.load_state_dict(torch.load(f"/content/out/lam1e-03.pt")); m_d.eval()
r_d=avaliar(m_d,held_n,n=min(24,len(held_n))); RES["diverso_no_estreito"]=r_d
a,b=r_n[-1],r_d[-1]
print(f"no MESMO held-out estreito:")
print(f"  treinado ESTREITO: {a['psnr']:.2f} dB @ {a['kb']:.2f} KB · JPEG {a['jpeg']:.2f} · Δ {a['psnr']-a['jpeg']:+.2f}")
print(f"  treinado DIVERSO : {b['psnr']:.2f} dB @ {b['kb']:.2f} KB · JPEG {b['jpeg']:.2f} · Δ {b['psnr']-b['jpeg']:+.2f}")
json.dump(RES,open("/content/out/resultados.json","w"),indent=1)


## 8) Resumo + reconstrução progressiva do melhor modelo


In [ ]:
best_k=max([k for k in RES],key=lambda k:RES[k][-1]["psnr"]-RES[k][-1]["jpeg"])
print("=== RESUMO (Δ vs JPEG no nível 4) ===")
for k in RES:
    f=RES[k][-1]; print(f"  {k:24s} {f['psnr']:6.2f} dB @ {f['kb']:6.2f} KB · JPEG {f['jpeg']:6.2f} · Δ {f['psnr']-f['jpeg']:+.2f}")
print("melhor:",best_k)
mb=PyramidN(SPECS).to(dev)
try: mb.load_state_dict(torch.load(f"/content/out/{ 'estreito' if best_k=='estreito' else 'lam1e-03'}.pt"))
except Exception: pass
mb.eval()
with torch.no_grad():
    im=held[0]; x=im[None].to(dev); o=mb(x,train=False)
    fig,ax=plt.subplots(1,len(SPECS)+1,figsize=(3.2*(len(SPECS)+1),3.4))
    ax[0].imshow(x[0].cpu().permute(1,2,0)); ax[0].set_title("original"); ax[0].axis("off")
    for L in range(len(SPECS)):
        r=o["recs"][L][0].cpu(); kb=sum(kb_of(z) for z in o["z"][:L+1])
        ax[L+1].imshow(r.permute(1,2,0).clamp(0,1)); ax[L+1].axis("off")
        ax[L+1].set_title(f"grau {L}: {psnr(r.numpy(),x[0].cpu().numpy()):.1f} dB / {kb:.1f} KB",fontsize=9)
    plt.tight_layout(); plt.show()
print("resultados em /content/out/resultados.json")


## 9) Se sobrar unidade (você tem ~70h)
- **modelo maior**: `SPECS=[(6,32),(5,24),(4,16),(3,12),(2,8)]` → 5 graus, mais detalhe fino
- **mais dados**: `N_IMGS=5000` (COCO inteiro)
- **vídeo**: I-frame + delta temporal (o M87 já deu **−32.6% vs JPEG** — é onde vocês mais ganham)
- **treino MUITO longo**: 200k passos num só λ, pra ver o teto real da arquitetura
